# 05 — Feature Extraction + PyTorch Grade Classifier

**Pipeline:**
1. Load labeled images from `datasets/unified/psa{1-10}/`
2. Extract two feature types per image:
   - **CV features** (20-dim): corner sharpness, edge quality, surface cleanliness, centering — from `grading_utils`
   - **CNN features** (512-dim): pretrained ResNet-18 backbone embeddings
3. Cache features to disk so extraction runs once
4. Train three MLP variants and compare:
   - `CV-MLP` — interpretable, fast, 20 inputs
   - `CNN-MLP` — powerful, 512 inputs
   - `Combined-MLP` — both fused, 532 inputs
5. Evaluate: confusion matrix, per-grade accuracy, feature importance
6. Save best model to `grade_mlp_best.pt`

**Label:** PSA grades 1–10 bucketed into 5 classes:

| Bucket | Grades | Name |
|--------|--------|------|
| 0 | 1–2 | Poor |
| 1 | 3–4 | Very Good |
| 2 | 5–6 | Excellent |
| 3 | 7–8 | Near Mint |
| 4 | 9–10 | Mint |

**Prerequisites:** Run `04_dataset_acquisition.ipynb` Section 29 to populate `datasets/unified/`.

In [ ]:
# ── Install + imports ─────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'torch', 'torchvision', 'tqdm', 'scikit-learn', 'matplotlib', 'numpy'])

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.models as models

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm.auto import tqdm
import json, pickle, warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120, 'axes.facecolor': '#0d1117',
    'figure.facecolor': '#0d1117', 'text.color': 'white',
    'axes.labelcolor': 'white', 'xtick.color': 'white', 'ytick.color': 'white',
})

# ── Paths & constants ─────────────────────────────────────────────
DATASET_DIR = Path('datasets/unified')
CACHE_DIR   = Path('datasets/feature_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PSA_TO_BUCKET = {1:0,2:0, 3:1,4:1, 5:2,6:2, 7:3,8:3, 9:4,10:4}
BUCKET_NAMES  = ['Poor\n(1-2)', 'VG\n(3-4)', 'Excellent\n(5-6)', 'NM\n(7-8)', 'Mint\n(9-10)']
NUM_CLASSES   = 5
CARD_H, CARD_W = 420, 300   # standard card crop size for CV features

DEVICE = torch.device('cuda'  if torch.cuda.is_available() else
                      'mps'   if torch.backends.mps.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

# ── Verify dataset exists ─────────────────────────────────────────
total = sum(len(list((DATASET_DIR / f'psa{g}').glob('*')))
            for g in range(1, 11) if (DATASET_DIR / f'psa{g}').exists())
if total == 0:
    print('⚠ datasets/unified/ is empty — run 04_dataset_acquisition.ipynb Section 29 first')
else:
    print(f'✅ Found {total:,} images in datasets/unified/')

## Step 1 — CV Feature Extractor

Extracts a **20-dimensional** feature vector per card image using classic computer vision.
All analysis functions operate on the raw pixel data — no neural network required.

| # | Feature | Source |
|---|---------|--------|
| 0 | Corner score (mean) | Harris + Laplacian sharpness on 4 corners |
| 1–4 | Per-corner sharpness (TL, TR, BL, BR) | Harris response |
| 5 | Edge score | Sobel gradient on border strips |
| 6–9 | Per-edge score (top, bottom, left, right) | Sobel |
| 10 | Surface score | Inverse scratch/stain density |
| 11 | Scratch density | Canny + morphology |
| 12 | Color uniformity | Std dev of HSV value channel |
| 13 | Stain score | Dark blob count |
| 14 | Centering score | Border symmetry |
| 15 | L–R centering ratio | Left/right border balance |
| 16 | T–B centering ratio | Top/bottom border balance |
| 17 | Brightness mean | Global luminance |
| 18 | Contrast (std) | Pixel value std dev |
| 19 | Blur score | Laplacian variance (sharpness proxy) |

In [ ]:
def extract_cv_features(img_path: Path, card_h=CARD_H, card_w=CARD_W) -> np.ndarray | None:
    """
    Extract a 20-dim CV feature vector from a card image.
    Returns None if the image can't be read.
    """
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.resize(img, (card_w, card_h))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h, w = gray.shape

    # ── Corner sharpness (features 0-4) ──────────────────────────
    corner_size = max(20, h // 14)
    corners = [
        gray[:corner_size, :corner_size],            # TL
        gray[:corner_size, w-corner_size:],          # TR
        gray[h-corner_size:, :corner_size],          # BL
        gray[h-corner_size:, w-corner_size:],        # BR
    ]
    corner_scores = []
    for patch in corners:
        lap = cv2.Laplacian(patch, cv2.CV_32F)
        corner_scores.append(float(np.clip(lap.var() / 500.0 * 100, 0, 100)))
    corner_mean = float(np.mean(corner_scores))

    # ── Edge quality (features 5-9) ───────────────────────────────
    border = max(8, h // 50)
    strips = {
        'top':    gray[:border, :],
        'bottom': gray[h-border:, :],
        'left':   gray[:, :border],
        'right':  gray[:, w-border:],
    }
    edge_scores = []
    for strip in strips.values():
        sobel = cv2.Sobel(strip.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
        uniformity = 1.0 - float(np.std(sobel) / (np.mean(np.abs(sobel)) + 1e-6)) / 10
        edge_scores.append(float(np.clip(uniformity * 100, 0, 100)))
    edge_mean = float(np.mean(edge_scores))

    # ── Surface quality (features 10-13) ─────────────────────────
    inner = gray[border*2:h-border*2, border*2:w-border*2]
    # Scratches: high-frequency linear features
    blur  = cv2.GaussianBlur(inner, (3, 3), 0)
    diff  = cv2.absdiff(inner, blur)
    scratch_density = float(np.sum(diff > 15) / diff.size)
    surface_score   = float(np.clip((1.0 - scratch_density * 20) * 100, 0, 100))
    # Color uniformity: std of V channel in inner region
    val_inner = hsv[border*2:h-border*2, border*2:w-border*2, 2]
    color_uniformity = float(np.std(val_inner.astype(np.float32)))
    # Stains: large dark blobs
    _, thresh = cv2.threshold(inner, 30, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    stain_count = float(sum(1 for c in contours if cv2.contourArea(c) > 100))
    stain_score = float(np.clip(100 - stain_count * 5, 0, 100))

    # ── Centering (features 14-16) ────────────────────────────────
    # Measure border widths via column/row mean projection
    col_proj = gray.mean(axis=0).astype(np.float32)
    row_proj = gray.mean(axis=1).astype(np.float32)
    thresh_val = float(col_proj.mean()) * 0.85

    def _border_width(proj, from_end=False):
        seq = proj[::-1] if from_end else proj
        for i, v in enumerate(seq):
            if v > thresh_val:
                return max(i, 1)
        return len(proj) // 4

    l = _border_width(col_proj)
    r = _border_width(col_proj, from_end=True)
    t = _border_width(row_proj)
    b = _border_width(row_proj, from_end=True)

    lr_ratio = float(min(l, r) / max(l, r)) if max(l, r) > 0 else 1.0
    tb_ratio = float(min(t, b) / max(t, b)) if max(t, b) > 0 else 1.0
    centering_score = float((lr_ratio * 0.5 + tb_ratio * 0.5) * 100)

    # ── Global image stats (features 17-19) ──────────────────────
    brightness = float(gray.mean())
    contrast   = float(gray.std())
    blur_score = float(cv2.Laplacian(gray, cv2.CV_32F).var())
    blur_score = float(np.clip(blur_score / 1000.0 * 100, 0, 100))

    return np.array([
        corner_mean, *corner_scores,                     # 0-4
        edge_mean,   *edge_scores,                       # 5-9
        surface_score, scratch_density * 100,            # 10-11
        color_uniformity, stain_score,                   # 12-13
        centering_score, lr_ratio * 100, tb_ratio * 100,# 14-16
        brightness, contrast, blur_score,                # 17-19
    ], dtype=np.float32)


FEATURE_NAMES = [
    'corner_mean', 'corner_TL', 'corner_TR', 'corner_BL', 'corner_BR',
    'edge_mean', 'edge_top', 'edge_bottom', 'edge_left', 'edge_right',
    'surface_score', 'scratch_density', 'color_uniformity', 'stain_score',
    'centering', 'lr_ratio', 'tb_ratio',
    'brightness', 'contrast', 'blur_score',
]
CV_DIM = len(FEATURE_NAMES)
print(f'✅ CV feature extractor ready — {CV_DIM} features per image')

# Quick smoke test
test_imgs = list(DATASET_DIR.glob('psa*/*.jpg'))[:1]
if test_imgs:
    fv = extract_cv_features(test_imgs[0])
    print(f'   Sample feature vector: {fv.round(1)}')

## Step 2 — CNN Feature Extractor

Uses a **frozen** pretrained ResNet-18 as a feature backbone.
The final classification head is removed; the avgpool output gives a 512-dim embedding.
No fine-tuning here — pure feature extraction.

In [ ]:
# ── Load frozen ResNet-18 backbone ───────────────────────────────
backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = nn.Identity()   # remove classification head → 512-dim output
backbone = backbone.to(DEVICE).eval()
for p in backbone.parameters():
    p.requires_grad = False

CNN_DIM = 512

cnn_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def extract_cnn_features(img_path: Path) -> np.ndarray | None:
    """Extract 512-dim ResNet-18 embedding from an image."""
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor  = cnn_transform(img_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb = backbone(tensor).squeeze(0).cpu().numpy()
    return emb.astype(np.float32)


print(f'✅ CNN feature extractor ready — {CNN_DIM}-dim ResNet-18 embeddings')
print(f'   Backbone frozen — no gradient updates during extraction')

## Step 3 — Extract & Cache All Features

Runs once over the full dataset. Features are saved to `datasets/feature_cache/`
so subsequent training runs load instantly without re-processing images.

In [ ]:
CACHE_FILE = CACHE_DIR / 'features.pkl'


def build_feature_cache(dataset_dir: Path = DATASET_DIR,
                        cache_file: Path  = CACHE_FILE,
                        force: bool       = False) -> dict:
    """
    Walk dataset_dir/psa{1-10}/, extract CV + CNN features for every image.
    Saves a dict with keys: paths, cv_features, cnn_features, grades, buckets.
    Loads from cache if it already exists.
    """
    if cache_file.exists() and not force:
        print(f'Loading cached features from {cache_file} …')
        with open(cache_file, 'rb') as f:
            cache = pickle.load(f)
        print(f'✅ Loaded {len(cache["paths"])} samples from cache')
        return cache

    print('Extracting features (runs once — results cached to disk) …')
    paths, cv_feats, cnn_feats, grades, buckets = [], [], [], [], []
    skipped = 0

    for grade in range(1, 11):
        folder = dataset_dir / f'psa{grade}'
        if not folder.exists():
            continue
        imgs = [p for p in sorted(folder.glob('*'))
                if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')]
        if not imgs:
            continue

        print(f'  PSA {grade:>2}  ({len(imgs)} images) …')
        for img_path in tqdm(imgs, desc=f'PSA {grade}', leave=False):
            cv  = extract_cv_features(img_path)
            cnn = extract_cnn_features(img_path)
            if cv is None or cnn is None:
                skipped += 1
                continue
            paths.append(str(img_path))
            cv_feats.append(cv)
            cnn_feats.append(cnn)
            grades.append(grade)
            buckets.append(PSA_TO_BUCKET[grade])

    cache = {
        'paths':        paths,
        'cv_features':  np.array(cv_feats,  dtype=np.float32),
        'cnn_features': np.array(cnn_feats, dtype=np.float32),
        'grades':       np.array(grades,    dtype=np.int32),
        'buckets':      np.array(buckets,   dtype=np.int32),
    }
    with open(cache_file, 'wb') as f:
        pickle.dump(cache, f)

    print(f'\n✅ Cached {len(paths)} samples  ({skipped} skipped)')
    print(f'   CV features  shape: {cache["cv_features"].shape}')
    print(f'   CNN features shape: {cache["cnn_features"].shape}')
    print(f'   Saved to: {cache_file}')
    return cache


# ── Run extraction (or load cache) ───────────────────────────────
cache = build_feature_cache()

if len(cache['paths']) == 0:
    print('\n⚠ No images found — run 04_dataset_acquisition.ipynb first')
else:
    # Grade distribution in cache
    from collections import Counter
    gc = Counter(int(g) for g in cache['grades'])
    print('\nGrade distribution:')
    for g in range(1, 11):
        n = gc.get(g, 0)
        print(f'  PSA {g:>2}: {n:>5}  {"█" * (n // max(max(gc.values()) // 20, 1))}')

## Step 4 — Dataset & DataLoaders

In [ ]:
class CardFeatureDataset(Dataset):
    """
    Dataset built from pre-extracted features.

    mode : 'cv'       → 20-dim CV features only
           'cnn'      → 512-dim CNN features only
           'combined' → 532-dim (CV + CNN concatenated)
    """
    def __init__(self, cache: dict, mode: str = 'combined',
                 indices=None, normalize: bool = True):
        self.mode   = mode
        self.labels = torch.tensor(cache['buckets'], dtype=torch.long)

        cv  = cache['cv_features']
        cnn = cache['cnn_features']

        if normalize:
            # Standardize each feature to mean=0, std=1 (use training stats)
            self.cv_mean  = cv.mean(0, keepdims=True)
            self.cv_std   = cv.std(0, keepdims=True)  + 1e-6
            self.cnn_mean = cnn.mean(0, keepdims=True)
            self.cnn_std  = cnn.std(0, keepdims=True) + 1e-6
            cv  = (cv  - self.cv_mean)  / self.cv_std
            cnn = (cnn - self.cnn_mean) / self.cnn_std

        if mode == 'cv':
            feats = cv
        elif mode == 'cnn':
            feats = cnn
        else:  # combined
            feats = np.concatenate([cv, cnn], axis=1)

        self.features = torch.tensor(feats, dtype=torch.float32)
        if indices is not None:
            self.features = self.features[indices]
            self.labels   = self.labels[indices]

    def __len__(self):  return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

    @property
    def input_dim(self):
        return self.features.shape[1]


def make_loaders(cache, mode='combined', val_frac=0.15,
                 batch_size=64, seed=42):
    """Stratified train/val split then DataLoaders."""
    import random
    rng = random.Random(seed)

    # Stratified split: keep grade distribution in both sets
    from collections import defaultdict
    bucket_idx = defaultdict(list)
    for i, b in enumerate(cache['buckets']):
        bucket_idx[int(b)].append(i)

    train_idx, val_idx = [], []
    for b, idxs in bucket_idx.items():
        rng.shuffle(idxs)
        split = max(1, int(len(idxs) * val_frac))
        val_idx   += idxs[:split]
        train_idx += idxs[split:]

    # Fit normalizer on train only, apply to both
    full_ds = CardFeatureDataset(cache, mode=mode, normalize=False)
    cv_mean  = cache['cv_features'][train_idx].mean(0, keepdims=True)
    cv_std   = cache['cv_features'][train_idx].std(0, keepdims=True)  + 1e-6
    cnn_mean = cache['cnn_features'][train_idx].mean(0, keepdims=True)
    cnn_std  = cache['cnn_features'][train_idx].std(0, keepdims=True) + 1e-6

    norm_cache = dict(cache)  # shallow copy
    norm_cache['cv_features']  = (cache['cv_features']  - cv_mean)  / cv_std
    norm_cache['cnn_features'] = (cache['cnn_features'] - cnn_mean) / cnn_std

    import numpy as np
    train_idx_arr = np.array(train_idx)
    val_idx_arr   = np.array(val_idx)

    train_ds = CardFeatureDataset(norm_cache, mode=mode, normalize=False, indices=train_idx_arr)
    val_ds   = CardFeatureDataset(norm_cache, mode=mode, normalize=False, indices=val_idx_arr)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)

    print(f'Mode: {mode} | input_dim: {train_ds.input_dim}')
    print(f'Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')
    return train_loader, val_loader, train_ds.input_dim


# Preview loaders for combined mode
if len(cache.get('paths', [])) > 0:
    _, _, _dim = make_loaders(cache, mode='combined')

## Step 5 — Model Architectures

Three MLP variants with the same interface — swap `mode` to choose features.

In [ ]:
class GradeMLP(nn.Module):
    """
    Configurable MLP for PSA grade bucket classification.

    input_dim  : CV_DIM (20), CNN_DIM (512), or CV_DIM+CNN_DIM (532)
    hidden_dims: list of hidden layer sizes
    dropout    : dropout probability after each hidden layer
    """
    def __init__(self, input_dim: int, num_classes: int = NUM_CLASSES,
                 hidden_dims=(256, 128, 64), dropout: float = 0.4):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            ]
            in_dim = h
        layers.append(nn.Linear(in_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Preset configs
MODEL_CONFIGS = {
    'CV-MLP': {
        'mode':        'cv',
        'hidden_dims': (64, 32),
        'dropout':     0.3,
        'lr':          3e-3,
        'epochs':      60,
    },
    'CNN-MLP': {
        'mode':        'cnn',
        'hidden_dims': (256, 128, 64),
        'dropout':     0.4,
        'lr':          1e-3,
        'epochs':      40,
    },
    'Combined-MLP': {
        'mode':        'combined',
        'hidden_dims': (512, 256, 128, 64),
        'dropout':     0.4,
        'lr':          1e-3,
        'epochs':      50,
    },
}

print('Model configs:')
for name, cfg in MODEL_CONFIGS.items():
    fake_dim = CV_DIM if cfg['mode']=='cv' else (CNN_DIM if cfg['mode']=='cnn' else CV_DIM+CNN_DIM)
    m = GradeMLP(fake_dim, hidden_dims=cfg['hidden_dims'], dropout=cfg['dropout'])
    print(f'  {name:<15} | input={fake_dim:>3} | params={m.count_params():,}')

## Step 6 — Training Loop

In [ ]:
def train_model(model, train_loader, val_loader,
                lr=1e-3, epochs=50, label='model') -> dict:
    """
    Train with Adam + CosineAnnealingLR + early stopping.
    Returns history dict with train/val loss and accuracy.
    """
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0
    best_state   = None
    patience     = 15
    no_improve   = 0

    for epoch in range(1, epochs + 1):
        # ── Train ───────────────────────────────────────────────
        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        for feats, labels in train_loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(feats)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            t_loss    += loss.item() * len(labels)
            t_correct += (out.argmax(1) == labels).sum().item()
            t_total   += len(labels)
        scheduler.step()

        # ── Validate ─────────────────────────────────────────────
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for feats, labels in val_loader:
                feats, labels = feats.to(DEVICE), labels.to(DEVICE)
                out  = model(feats)
                loss = criterion(out, labels)
                v_loss    += loss.item() * len(labels)
                v_correct += (out.argmax(1) == labels).sum().item()
                v_total   += len(labels)

        t_acc = t_correct / t_total
        v_acc = v_correct / v_total
        history['train_loss'].append(t_loss / t_total)
        history['val_loss'].append(v_loss / v_total)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

        if epoch % 10 == 0 or epoch == 1:
            print(f'  [{label}] Epoch {epoch:>3}/{epochs}  '
                  f'train_acc={t_acc:.3f}  val_acc={v_acc:.3f}')

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve   = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {epoch} (best val_acc={best_val_acc:.3f})')
                break

    model.load_state_dict(best_state)
    history['best_val_acc'] = best_val_acc
    return history


print('✅ Training loop ready')

## Step 7 — Train All Three Models

In [ ]:
if len(cache.get('paths', [])) == 0:
    print('⚠ No features cached — run the dataset acquisition notebook first')
else:
    trained_models  = {}
    trained_loaders = {}
    all_history     = {}

    for name, cfg in MODEL_CONFIGS.items():
        print(f'\n{'─'*55}')
        print(f'  Training {name}')
        print(f'{'─'*55}')

        train_loader, val_loader, in_dim = make_loaders(
            cache, mode=cfg['mode'], batch_size=128
        )
        model = GradeMLP(
            input_dim   = in_dim,
            hidden_dims = cfg['hidden_dims'],
            dropout     = cfg['dropout'],
        )
        history = train_model(
            model, train_loader, val_loader,
            lr=cfg['lr'], epochs=cfg['epochs'], label=name,
        )
        trained_models[name]  = model
        trained_loaders[name] = (train_loader, val_loader)
        all_history[name]     = history
        print(f'  ✅ {name} best val_acc = {history["best_val_acc"]:.3f}')

    print(f'\n{'═'*55}')
    print('  Summary')
    print(f'{'═'*55}')
    for name, h in all_history.items():
        print(f'  {name:<15}  val_acc = {h["best_val_acc"]:.3f}')

## Step 8 — Evaluation: Training Curves + Confusion Matrix

In [ ]:
if not trained_models:
    print('Train models first (Step 7)')
else:
    from sklearn.metrics import confusion_matrix

    n_models = len(trained_models)
    fig = plt.figure(figsize=(6 * n_models, 10))
    gs  = gridspec.GridSpec(2, n_models, figure=fig, hspace=0.4)

    for col, (name, model) in enumerate(trained_models.items()):
        h = all_history[name]
        _, val_loader = trained_loaders[name]

        # ── Training curve ────────────────────────────────────────
        ax1 = fig.add_subplot(gs[0, col])
        epochs_range = range(1, len(h['train_acc']) + 1)
        ax1.plot(epochs_range, h['train_acc'], color='#4ade80', label='train')
        ax1.plot(epochs_range, h['val_acc'],   color='#fb923c', label='val', linestyle='--')
        ax1.set_title(f'{name}\nbest val={h["best_val_acc"]:.3f}', color='white')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
        ax1.legend(); ax1.set_ylim(0, 1)

        # ── Confusion matrix ──────────────────────────────────────
        ax2 = fig.add_subplot(gs[1, col])
        model.eval()
        all_preds, all_true = [], []
        with torch.no_grad():
            for feats, labels in val_loader:
                preds = model(feats.to(DEVICE)).argmax(1).cpu()
                all_preds.extend(preds.tolist())
                all_true.extend(labels.tolist())

        cm = confusion_matrix(all_true, all_preds, labels=list(range(NUM_CLASSES)))
        cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

        im = ax2.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
        tick_labels = ['Poor', 'VG', 'Exc.', 'NM', 'Mint']
        ax2.set_xticks(range(NUM_CLASSES)); ax2.set_xticklabels(tick_labels, fontsize=8)
        ax2.set_yticks(range(NUM_CLASSES)); ax2.set_yticklabels(tick_labels, fontsize=8)
        ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')
        ax2.set_title('Confusion Matrix (normalized)', color='white')
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                txt = f'{cm_norm[i,j]:.2f}'
                ax2.text(j, i, txt, ha='center', va='center',
                         color='white' if cm_norm[i,j] > 0.5 else 'black', fontsize=7)

    plt.suptitle('Model Evaluation', color='white', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## Step 9 — CV Feature Importance

Which of the 20 CV features matter most for predicting grade?
Computed via **permutation importance** on the CV-MLP model.

In [ ]:
if 'CV-MLP' not in trained_models:
    print('Train CV-MLP first')
else:
    cv_model = trained_models['CV-MLP'].eval()
    _, val_loader = trained_loaders['CV-MLP']

    # Collect all val features + labels
    all_feats, all_labels = [], []
    for feats, labels in val_loader:
        all_feats.append(feats)
        all_labels.append(labels)
    all_feats  = torch.cat(all_feats).to(DEVICE)
    all_labels = torch.cat(all_labels).to(DEVICE)

    def accuracy(feats, labels):
        with torch.no_grad():
            preds = cv_model(feats).argmax(1)
        return (preds == labels).float().mean().item()

    base_acc = accuracy(all_feats, all_labels)
    importances = []
    for i in range(CV_DIM):
        shuffled = all_feats.clone()
        idx = torch.randperm(shuffled.shape[0])
        shuffled[:, i] = shuffled[idx, i]  # permute column i
        drop = base_acc - accuracy(shuffled, all_labels)
        importances.append(drop)

    # Plot
    imp = np.array(importances)
    order = np.argsort(imp)[::-1]

    fig, ax = plt.subplots(figsize=(12, 4))
    colors = ['#22c55e' if imp[i] > 0 else '#ef4444' for i in order]
    ax.bar(range(CV_DIM), imp[order], color=colors, edgecolor='#30363d')
    ax.set_xticks(range(CV_DIM))
    ax.set_xticklabels([FEATURE_NAMES[i] for i in order],
                       rotation=45, ha='right', fontsize=8)
    ax.axhline(0, color='white', linewidth=0.5)
    ax.set_ylabel('Accuracy drop when permuted')
    ax.set_title(f'CV Feature Importance  (base_acc={base_acc:.3f})', color='white')
    plt.tight_layout()
    plt.show()

    print('Top 5 most important CV features:')
    for rank, i in enumerate(order[:5]):
        print(f'  {rank+1}. {FEATURE_NAMES[i]:<22}  drop={imp[i]:+.4f}')

## Step 10 — Save Best Model

In [ ]:
if not trained_models:
    print('No models trained yet')
else:
    # Pick model with best val accuracy
    best_name = max(all_history, key=lambda n: all_history[n]['best_val_acc'])
    best_model = trained_models[best_name]
    best_acc   = all_history[best_name]['best_val_acc']
    cfg = MODEL_CONFIGS[best_name]

    save_path = Path('grade_mlp_best.pt')
    torch.save({
        'model_state': best_model.state_dict(),
        'model_name':  best_name,
        'mode':        cfg['mode'],
        'hidden_dims': cfg['hidden_dims'],
        'dropout':     cfg['dropout'],
        'input_dim':   best_model.net[0].in_features,
        'num_classes': NUM_CLASSES,
        'val_acc':     best_acc,
        'bucket_names': BUCKET_NAMES,
        'feature_names': FEATURE_NAMES,   # only relevant for CV/Combined modes
    }, save_path)

    print(f'✅ Best model: {best_name}  (val_acc={best_acc:.3f})')
    print(f'   Saved to: {save_path}')
    print()
    print('To reload and predict:')
    print("  ckpt  = torch.load('grade_mlp_best.pt')")
    print("  model = GradeMLP(ckpt['input_dim'], hidden_dims=ckpt['hidden_dims'])")
    print("  model.load_state_dict(ckpt['model_state'])")
    print("  model.eval()")